# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id
record_sets = metadata.recordSet
if not record_sets:
    print("No record sets found in the metadata. Please check the Croissant schema.")
else:
    for rs in record_sets:
        print(f"RecordSet: @id={rs['@id']}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for field in fields:
                if isinstance(field, dict):
                    print(f"  Field: @id={field.get('@id', 'N/A')}, name={field.get('name', 'N/A')}, dataType={field.get('dataType', 'N/A')}")
                else:
                    print(f"  Field: @id={field}")
        else:
            print("  No fields found for this record set.")
    
    # Example: Show a sample record from the first record set
    print('\nSample record from the first record set (if available):')
    first_rs_id = record_sets[0]['@id']
    records_iter = dataset.records(record_set=first_rs_id)
    first_record = next(records_iter, None)
    print(first_record)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into DataFrames
record_sets = metadata.recordSet
dataframes = {}

# Store record set @ids for reference
record_set_ids = []
if not record_sets:
    print('No record sets available to extract.')
else:
    # In this dataset, recordSet may be empty. If so, try loading from a known 
    # data package or find the recordSet ids from the Croissant metadata.
    for rs in record_sets:
        rs_id = rs['@id']
        record_set_ids.append(rs_id)
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[rs_id])} records from RecordSet @id={rs_id}")
        else:
            print(f"No records found for RecordSet @id={rs_id}")

    # Display columns for the first available record set
    if len(dataframes) > 0:
        first_rs_id = list(dataframes.keys())[0]
        print(f"\nColumns in DataFrame for RecordSet @id={first_rs_id}:")
        print(dataframes[first_rs_id].columns.tolist())
        display(dataframes[first_rs_id].head())
    else:
        print("No DataFrames to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a DataFrame/RecordSet for analysis (adjust as needed)
if dataframes:
    # Choose first available record set and guess a numeric field (replace with exact @id as needed):
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using DataFrame from RecordSet @id={record_set_id}")
    
    # Attempt to find a numeric field from the DataFrame
    numeric_field_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_field_candidates:
        print('No numeric fields found for analysis.')
    else:
        numeric_field = numeric_field_candidates[0]
        print(f"Using numeric field '{numeric_field}' for EDA.")
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Grouping by a string/categorical field if available
        group_field_candidates = df.select_dtypes(include=['object']).columns.tolist()
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print('No suitable group field found.')
else:
    print('No DataFrame available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(10, 5))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouping exists
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Visualization skipped: No suitable numeric field.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!-- Add your summary and reflections on the dataset insights here. -->